# **Token Classification using BERT (NER Task)**

This project implements a token classification system using a Transformer model (BERT) for Named Entity Recognition (NER).

We use the WikiANN dataset and perform:
- Data preprocessing
- Tokenization and label alignment
- Model training using Hugging Face Trainer
- Evaluation using seqeval
- Inference on custom sentences

Pipeline:
Raw Data → Tokenization → Label Alignment → Training → Evaluation → Inference

## **Install Dependencies**

In [8]:
!pip install -U transformers datasets evaluate seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 106.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      Successfully uninstalled hf-xet-1.4.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: 

## **Import Libraries**

In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
import evaluate

## **Task 1: Dataset Selection**


We use the WikiANN dataset for Named Entity Recognition (Chunking task).
It contains BIO-tagged entities such as Person, Organization, and Location.

In [ ]:
dataset = load_dataset("wikiann", "en")
print(dataset)

### **Label Categories**

The dataset uses BIO tagging format:
- B-PER, I-PER → Person
- B-ORG, I-ORG → Organization
- B-LOC, I-LOC → Location
- O → Outside

In [ ]:
label_list = dataset["train"].features["ner_tags"].feature.names
print(label_list)

## **Task 2: Tokenization and Label Alignment**

We use BERT tokenizer and handle:
- Subword tokens
- Special tokens using -100

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

## **Tokenization Function**

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## **Apply Preprocessing**

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

## **Task 3: Model Setup**

We use BERT with token classification head.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label={i: label for i, label in enumerate(label_list)},
    label2id={label: i for i, label in enumerate(label_list)}
)

## **Task 4: Training**

We train the model using Hugging Face Trainer.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
)

## **Data Collator**

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

## **Task 5: Evaluation**

We use seqeval to compute Precision, Recall, and F1 score.

In [ ]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

## **Trainer Setup**

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## **Train Model**

In [ ]:
trainer.train()

## **Evaluate**

In [ ]:
trainer.evaluate()

## **Task 6: Inference**

We test the model on a custom sentence.

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)

sentence = "Abhay is Intern at Innomatics Research Labs"
results = ner_pipeline(sentence)

for r in results:
    print(r)

## Task 7: Comparison between POS Tagging and Chunking

### Example Sentence:
Abhay is Intern at Innomatics Research Labs

---

### 1. POS Tagging Output (Grammar-Level)

| Word | POS Tag | Meaning |
|------|--------|--------|
| Abhay | PROPN | Proper Noun |
| is | AUX | Auxiliary Verb |
| Intern | NOUN | Noun |
| at | ADP | Preposition |
| Innomatics | PROPN | Proper Noun |
| Research | PROPN | Proper Noun |
| Labs | PROPN | Proper Noun |

Explanation:
POS tagging assigns grammatical roles to each word in the sentence.
It helps in understanding the structure of the sentence.

---

### 2. Chunking Output (Entity-Level / NER)

| Word | Chunk Tag | Meaning |
|------|----------|--------|
| Abhay | B-PER | Beginning of Person |
| Innomatics | B-ORG | Beginning of Organization |
| Research | I-ORG | Inside Organization |
| Labs | I-ORG | Inside Organization |

Explanation:
Chunking identifies meaningful entities in the sentence.
“Innomatics Research Labs” is recognized as a single organization.

---

### 3. Key Differences

| Feature | POS Tagging | Chunking |
|--------|------------|----------|
| Level | Word-level | Phrase-level |
| Focus | Grammar | Meaning |
| Output | NOUN, VERB | B-PER, B-ORG |
| Complexity | Easy | Medium |
| Context Usage | Low | High |

---

### 4. Conclusion

POS tagging focuses on identifying grammatical roles of words, whereas chunking focuses on extracting meaningful entities from text.

In this example:
- POS tagging identifies "Abhay" as a proper noun.
- Chunking identifies "Abhay" as a person entity.
- Chunking also groups "Innomatics Research Labs" as a single organization, which POS tagging cannot do.

Thus, chunking provides deeper semantic understanding compared to POS tagging.